# Exceptions et tests

Quand on code avec un LLM ou seul, on fait face à deux types de situations :

1. **Le code plante** — il faut lire l'erreur et décider quoi en faire
2. **Le code tourne mais donne un résultat faux** — il faut des gardes-fous

Ce notebook couvre les trois outils Python pour gérer ça :
- `try / except` — intercepter les erreurs prévisibles
- `assert` — vérifier des hypothèses sur ses données
- `pytest` — tester ses fonctions de façon reproductible

---
## 1. Lire un traceback — ne pas paniquer

Un traceback Python se lit **de bas en haut** : la dernière ligne indique l'erreur, les lignes au-dessus montrent la pile d'appels.

```
Traceback (most recent call last):       ← ignorer ce header
  File "script.py", line 12, in <module> ← où ça a planté
    result = diviser(10, 0)              ← la ligne fautive
  File "script.py", line 5, in diviser  ← dans quelle fonction
    return a / b
ZeroDivisionError: division by zero      ← COMMENCER ICI
```

**Méthode** : lire le type d'erreur + message (dernière ligne), puis remonter pour trouver la ligne de code responsable.

### 1.1 Les erreurs les plus fréquentes

| Erreur | Cause typique |
|---|---|
| `TypeError` | Opération sur un type inattendu (`3 + "hello"`) |
| `KeyError` | Clé absente dans un dictionnaire ou DataFrame |
| `IndexError` | Index hors bornes dans une liste |
| `ValueError` | Valeur incompatible (`int("abc")`) |
| `AttributeError` | Méthode inexistante sur un objet |
| `FileNotFoundError` | Fichier introuvable |
| `ImportError` | Package non installé |
| `ZeroDivisionError` | Division par zéro |

⚠️ **Prédit avant d'exécuter** : quel type d'erreur lève chacune de ces cellules ?

In [ ]:
3 + "hello"

In [ ]:
d = {"a": 1}
d["b"]

In [ ]:
int("abc")

---
## 2. `try / except` — gérer les erreurs attendues

À utiliser quand une erreur est **prévisible** (réseau, fichier absent, type inconnu d'un utilisateur) et qu'on veut que le programme continue malgré tout.

### 2.1 Syntaxe de base

In [ ]:
try:
    result = 3 + "hello"   # lève un TypeError
except:
    print("quelque chose s'est mal passé")

print("le programme continue")

> **Ne jamais utiliser `except:` seul** (bare except). Il masque toutes les erreurs y compris les bugs inattendus. Toujours préciser le type d'exception.

### 2.2 Attraper une exception précise

In [ ]:
try:
    result = 3 + "hello"
except TypeError as e:
    print(f"Erreur de type : {e}")

print("le programme continue")

⚠️ **Prédit avant d'exécuter** : ce code affiche-t-il « Erreur de type » ou plante-t-il ?

In [ ]:
# On attrape TypeError mais l'erreur réelle est NameError
try:
    print(variable_non_definie)
except TypeError as e:
    print(f"Erreur de type : {e}")

print("le programme continue")

L'exception non attrapée remonte et plante le programme. **C'est le comportement voulu** : seules les erreurs prévues sont interceptées.

Pour attraper plusieurs types d'exceptions :

In [ ]:
try:
    print(variable_non_definie)
except (TypeError, NameError) as e:
    print(f"Erreur : {repr(e)}")

### 2.3 `else`, `finally` et `raise`

La syntaxe complète d'un bloc try :

In [ ]:
try:
    result = 10 / 2
except ZeroDivisionError as e:
    print(f"Impossible : {e}")
else:
    print(f"Succès : {result}")   # exécuté SEULEMENT si pas d'exception
finally:
    print("Toujours exécuté (succès ou échec)")  # nettoyage, logs…

**`raise`** permet de lever une exception soi-même — utile pour signaler qu'un argument est invalide :

In [ ]:
def diviser(a, b):
    if b == 0:
        raise ValueError("Le diviseur ne peut pas être zéro")
    return a / b

diviser(10, 0)

### 2.4 Cas d'usage data science : lire un fichier avec cache

Pattern classique : charger un fichier depuis le disque s'il existe, le télécharger sinon.

In [ ]:
import pandas as pd
import os

URL = "https://raw.githubusercontent.com/pcm-dpc/COVID-19/master/dati-province/dpc-covid19-ita-province-20200225.csv"
CACHE_PATH = "./data/covid_italie.csv"

def charger_avec_cache(url, cache_path):
    """Charge un CSV depuis le cache local, ou le télécharge si absent."""
    try:
        df = pd.read_csv(cache_path)
        print(f"Chargé depuis le cache : {cache_path}")
    except FileNotFoundError:
        print("Cache absent — téléchargement...")
        df = pd.read_csv(url)
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        df.to_csv(cache_path, index=False)
        print(f"Enregistré dans : {cache_path}")
    return df

df = charger_avec_cache(URL, CACHE_PATH)
df.shape

> On attrape `FileNotFoundError` précisément : si le fichier existe mais est corrompu, l'erreur remonte — c'est voulu.

---
## 3. `assert` — valider ses hypothèses

`assert <condition>` lève une `AssertionError` si la condition est fausse. C'est un **contrat** : « cette hypothèse doit être vraie pour que la suite ait un sens ».

In [ ]:
assert 2 + 2 == 4          # pas d'erreur
assert isinstance(42, int)  # pas d'erreur

In [ ]:
assert 2 + 2 == 5, "Les mathématiques ont changé"

### 3.1 Cas d'usage data science

Valider des hypothèses sur les données — particulièrement après des jointures ou des transformations :

In [ ]:
import pandas as pd

# Simuler deux tables à joindre
commandes = pd.DataFrame({"id_client": [1, 2, 3, 2], "montant": [100, 200, 150, 80]})
clients   = pd.DataFrame({"id_client": [1, 2, 3],    "nom": ["Alice", "Bob", "Carol"]})

resultat = commandes.merge(clients, on="id_client", how="left")

# Vérifier les hypothèses après jointure
assert len(resultat) == len(commandes), "La jointure a changé le nombre de lignes !"
assert resultat["nom"].notna().all(),  "Des clients non trouvés dans la table clients !"

resultat

### 3.2 `assert` vs `try/except` — quand utiliser quoi ?

| | `assert` | `try / except` |
|---|---|---|
| **Usage** | Vérifier des invariants internes, valider des données | Gérer des erreurs extérieures prévisibles |
| **Si ça échoue** | Bug dans le code ou données incorrectes → on veut savoir | Erreur normale (réseau, fichier absent) → on s'en remet |
| **Exemple** | `assert len(df) > 0` après un chargement | `except FileNotFoundError` |

> ⚠️ Les `assert` sont **désactivés** si Python est lancé avec le flag `-O` (optimisation). Ne jamais utiliser `assert` pour des vérifications de sécurité.

---
## 4. Tests avec pytest — pour les fonctions réutilisées

Dès qu'on écrit une fonction qu'on va réutiliser (pipeline de données, transformation), écrire des tests permet de :
- **Vérifier que ça marche** maintenant
- **Détecter les régressions** plus tard (modification accidentelle)
- **Documenter le comportement attendu** pour les collègues (et les LLMs)

On utilise `ipytest` pour lancer pytest directement depuis un notebook.

In [ ]:
%pip install ipytest -q

In [ ]:
import ipytest
ipytest.autoconfig()

### 4.1 TDD — écrire le test avant le code

Le **Test-Driven Development** (TDD) consiste à écrire les tests *d'abord*, puis à écrire le code qui les fait passer. Avantage : on est obligé de définir le comportement attendu avant de coder.

**Exemple** : une fonction qui normalise un montant entre 0 et 1.

In [ ]:
%%ipytest

# On écrit les tests AVANT la fonction
def test_normaliser_min():
    assert normaliser([0, 50, 100], 0) == 0.0

def test_normaliser_max():
    assert normaliser([0, 50, 100], 100) == 1.0

def test_normaliser_milieu():
    assert normaliser([0, 50, 100], 50) == 0.5

def test_normaliser_valeur_hors_plage():
    # une valeur hors plage peut être > 1 ou < 0
    assert normaliser([0, 50, 100], 200) == 2.0

In [ ]:
def normaliser(serie, valeur):
    """Normalise une valeur par rapport au min/max d'une série."""
    mini, maxi = min(serie), max(serie)
    return (valeur - mini) / (maxi - mini)

In [ ]:
%%ipytest

def test_normaliser_min():
    assert normaliser([0, 50, 100], 0) == 0.0

def test_normaliser_max():
    assert normaliser([0, 50, 100], 100) == 1.0

def test_normaliser_milieu():
    assert normaliser([0, 50, 100], 50) == 0.5

def test_normaliser_valeur_hors_plage():
    assert normaliser([0, 50, 100], 200) == 2.0

### 4.2 `pytest.mark.parametrize` — tester plusieurs cas en une fois

Plutôt que d'écrire une fonction de test par cas, `parametrize` factorise :

In [ ]:
%%ipytest
import pytest

@pytest.mark.parametrize("valeur, attendu", [
    (0,   0.0),
    (50,  0.5),
    (100, 1.0),
    (200, 2.0),
    (-50, -0.5),
])
def test_normaliser(valeur, attendu):
    assert normaliser([0, 50, 100], valeur) == attendu

### 4.3 Tester les exceptions avec `pytest.raises`

Pour vérifier qu'une fonction lève bien l'erreur attendue :

In [ ]:
%%ipytest
import pytest

def test_normaliser_serie_vide():
    with pytest.raises((ValueError, ZeroDivisionError)):
        normaliser([], 5)   # min/max d'une liste vide → erreur

---
## 5. Récapitulatif : quel outil pour quel besoin ?

| Situation | Outil |
|---|---|
| Fichier parfois absent, réseau instable, type d'entrée inconnu | `try / except` |
| Vérifier qu'un DataFrame a le bon nombre de lignes après un merge | `assert` |
| Vérifier qu'une colonne n'a pas de NaN après nettoyage | `assert` |
| Valider une fonction utilitaire réutilisée dans un pipeline | `pytest` |
| Signaler qu'un argument est invalide dans une fonction | `raise ValueError(...)` |
| Garantir qu'un bloc de nettoyage (fermeture fichier, log) s'exécute toujours | `finally` |

---
## 6. Quiz

**Q1.** Ce code affiche-t-il quelque chose ou plante-t-il ?

```python
try:
    x = int("bonjour")
except TypeError:
    print("TypeError")
```

A) Affiche `TypeError`  
B) Plante avec une `ValueError`  
C) Rien ne s'affiche

<details><summary>Réponse</summary>

**B) Plante avec une `ValueError`**

`int("bonjour")` lève une `ValueError` (pas une `TypeError`). Le bloc `except TypeError` ne l'intercepte pas, donc l'erreur remonte.

</details>

**Q2.** Que fait ce code ? Est-ce une bonne pratique ?

```python
try:
    df = pd.read_csv("donnees.csv")
    df["ratio"] = df["ventes"] / df["stock"]
except:
    print("erreur")
```

A) Bonne pratique — attrape toutes les erreurs possibles  
B) Mauvaise pratique — masque des erreurs distinctes (fichier absent vs division par zéro vs colonne manquante)  
C) Ce code ne compile pas

<details><summary>Réponse</summary>

**B) Mauvaise pratique**

Le `except:` nu attrape tout — y compris `KeyboardInterrupt`. On ne sait plus si le problème vient du fichier, d'une colonne manquante ou d'une division par zéro. Toujours préciser le type d'exception.

</details>

**Q3.** Après ce code, `df_final` a-t-il plus ou moins de lignes que `df_commandes` ?

```python
df_commandes = pd.DataFrame({"id": [1, 2, 3, 1], "montant": [100, 200, 300, 50]})
df_clients   = pd.DataFrame({"id": [1, 2],        "nom": ["Alice", "Bob"]})
df_final = df_commandes.merge(df_clients, on="id", how="left")
assert len(df_final) == len(df_commandes)
```

A) Moins — les lignes sans correspondance sont supprimées  
B) Autant — l'assert passe  
C) L'assert lève une erreur

<details><summary>Réponse</summary>

**B) Autant — l'assert passe**

Un `merge` avec `how="left"` conserve toutes les lignes du DataFrame de gauche. La ligne avec `id=3` (absent de `df_clients`) obtient `NaN` pour la colonne `nom`, mais la ligne est conservée. L'assert passe — mais on devrait aussi ajouter `assert df_final["nom"].notna().all()` pour détecter les clients non trouvés.

</details>

**Q4.** Le bloc `finally` s'exécute-t-il dans les deux cas ?

```python
def lire(path):
    try:
        df = pd.read_csv(path)
        return df
    except FileNotFoundError:
        print("Fichier absent")
        return None
    finally:
        print("Fin de lire()")
```

A) Non — `finally` ne s'exécute pas quand `return` est atteint dans `try`  
B) Non — `finally` ne s'exécute pas quand une exception est attrapée  
C) Oui — `finally` s'exécute toujours, avant le `return`

<details><summary>Réponse</summary>

**C) Oui — `finally` s'exécute toujours**

`finally` est garanti de s'exécuter qu'il y ait eu une exception ou non, même quand un `return` est atteint dans `try` ou `except`. C'est son utilité principale : fermer des fichiers, écrire des logs, libérer des ressources.

</details>

**Q5.** Quelle est la différence entre ces deux approches pour valider des données ?

```python
# Approche A
assert df["age"].min() >= 0, "Des âges négatifs détectés"

# Approche B
try:
    if df["age"].min() < 0:
        raise ValueError("Des âges négatifs détectés")
except ValueError as e:
    print(e)
```

A) Elles sont équivalentes  
B) L'approche A plante le programme, l'approche B affiche le message et continue  
C) L'approche B est toujours préférable

<details><summary>Réponse</summary>

**B) L'approche A plante le programme, l'approche B affiche le message et continue**

En data science dans un notebook, les deux ont leur place :
- `assert` est idéal pendant l'exploration : on veut que le notebook s'arrête si les données sont anormales (sinon toutes les cellules suivantes calculent sur des données corrompues).
- `try/except` est adapté en production où on veut logger l'erreur et continuer le traitement des autres fichiers/lignes.

</details>